# BRSR RAG Pipeline — End-to-End with Retrieval Benchmarking

This notebook implements a full **Retrieval-Augmented Generation (RAG)** pipeline over BRSR (Business Responsibility and Sustainability Reports) for 20 Indian listed companies, together with a rigorous retrieval evaluation benchmark.

---

## Pipeline Overview

| Step | Description | Output |
|------|-------------|--------|
| 1 | **PDF Extraction** | Raw page text per company |
| 2 | **Chunking** | Two chunk-size strategies (300 & 800 tokens) |
| 3 | **Embedding** | BGE-small / BGE-base vector representations |
| 4 | **FAISS Indexing** | Cosine-similarity vector index |
| 5 | **Retrieval** | Company-scoped top-k retrieval |
| 6 | **LLM Generation** | Gemini-powered answer synthesis (optional) |
| 7 | **Benchmarking** | Precision@k, Recall@k, MRR across 2 setups |

---

## Directory Layout

```
project_root/
├── DATA/                  ← Input PDF files (one per company)
├── outputs/               ← All generated artefacts
│   ├── brsr_pages.jsonl
│   ├── brsr_chunks_strategy_{A,B}.jsonl
│   ├── benchmarks/
│   │   └── <setup_name>/
│   │       ├── index.faiss
│   │       ├── metadata.parquet
│   │       ├── vectors.npy
│   │       ├── evaluation_rows.jsonl
│   │       ├── failure_cases.jsonl
│   │       └── summary.json
│   ├── evaluation_retrieval_metrics.jsonl
│   └── ground_truth_raw.json
```



## 0 · Install Dependencies

In [1]:

!pip install pdfplumber langchain langchain-text-splitters sentence-transformers faiss-cpu tiktoken google-generativeai tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1 · Imports & Configuration

In [2]:
from __future__ import annotations

import ast
import json
import os
import re
from pathlib import Path
from typing import Any

import faiss
import numpy as np
import pandas as pd
import pdfplumber
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "DATA"
OUT_DIR  = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Chunking strategies ──────────────────────────────────────────────────────
CHUNK_STRATEGIES = {
    "A": {"chunk_size": 300, "chunk_overlap": 50},
    "B": {"chunk_size": 800, "chunk_overlap": 100},
}

# ── Benchmarking setups ───────────────────────────────────────────────────────
SETUPS: dict[str, dict[str, str]] = {
    "300_tokens_bge_small": {"strategy": "A", "embedding_model": "BAAI/bge-small-en-v1.5"},
    "800_tokens_bge_small": {"strategy": "B", "embedding_model": "BAAI/bge-small-en-v1.5"}
}

# ── Retrieval / evaluation parameters ────────────────────────────────────────
TOP_K               = 5
RELEVANCE_THRESHOLD = 0.45   # cosine-sim threshold for a chunk to be "relevant"
DEFAULT_GEMINI_MODEL = "gemma-3-27b-it"

# ── Tokeniser (shared across all chunking) ───────────────────────────────────
_ENCODING = tiktoken.get_encoding("cl100k_base")

def token_len(text: str) -> int:
    return len(_ENCODING.encode(text or ""))

print("Configuration loaded.")
print(f"  DATA_DIR : {DATA_DIR}")
print(f"  OUT_DIR  : {OUT_DIR}")
print(f"  Setups   : {list(SETUPS.keys())}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration loaded.
  DATA_DIR : /Users/saudquadri/Desktop/Godrej/BRSR_RAG/DATA
  OUT_DIR  : /Users/saudquadri/Desktop/Godrej/BRSR_RAG/outputs
  Setups   : ['300_tokens_bge_small', '800_tokens_bge_small']


---
## 2 · PDF Extraction

Extract page-level text from every PDF in `DATA/`.  
The company name is inferred from the filename prefix (everything before the first `_`).

**Output:** `outputs/brsr_pages.jsonl`

In [6]:
def company_name_from_filename(pdf_path: Path) -> str:
    """Return the ticker/code prefix before the first underscore in the filename."""
    stem = pdf_path.stem
    return stem.split("_", 1)[0] if "_" in stem else stem


def extract_pdf_pages(pdf_path: Path) -> list[dict[str, Any]]:
    """Extract text from every page of a single PDF."""
    company = company_name_from_filename(pdf_path)
    rows: list[dict[str, Any]] = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = (page.extract_text() or "").replace("\x00", "").strip("\n")
            rows.append({
                "company_name": company,
                "file_name":    pdf_path.name,
                "page_number":  page_num,
                "page_text":    text,
            })
    return rows


def extract_all_pdfs(data_dir: Path) -> pd.DataFrame:
    """Extract pages from all PDFs in *data_dir* and return a DataFrame."""
    assert data_dir.exists(), f"DATA folder not found: {data_dir}"
    pdf_files = sorted(data_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDFs in {data_dir}")

    rows: list[dict[str, Any]] = []
    for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
        rows.extend(extract_pdf_pages(pdf_path))

    df = pd.DataFrame(rows)
    empty = df["page_text"].str.strip().eq("").sum()
    print(f"Total pages  : {len(df)}")
    print(f"Companies    : {df['company_name'].nunique()}")
    print(f"Empty pages  : {empty} / {len(df)}")
    return df


# ── Run extraction ────────────────────────────────────────────────────────────
pages_path = OUT_DIR / "brsr_pages.jsonl"

if pages_path.exists():
    print(f"Loading cached extraction from {pages_path}")
    pages_df = pd.read_json(pages_path, lines=True)
else:
    pages_df = extract_all_pdfs(DATA_DIR)
    pages_df.to_json(pages_path, orient="records", lines=True, force_ascii=False)
    print(f"Saved: {pages_path}")

pages_df.head(3)

Loading cached extraction from /Users/saudquadri/Desktop/Godrej/BRSR_RAG/outputs/brsr_pages.jsonl


,company_name,file_name,page_number,page_text
0,ASIANPAINT,ASIANPAINT_24062025100951_BRSRIntimation030620...,1,APL/SEC/26/2025-26/14\n3rd June 2025\nBSE Limi...
1,ASIANPAINT,ASIANPAINT_24062025100951_BRSRIntimation030620...,2,Statutory reports\nBusiness Responsibility and...
2,ASIANPAINT,ASIANPAINT_24062025100951_BRSRIntimation030620...,3,Business Responsibility and Sustainability Rep...


---
## 3 · Chunking

Each company's pages are concatenated into one document, then split into overlapping token-bounded chunks.

Two strategies are produced:

| Strategy | Chunk size | Overlap |
|----------|------------|--------|
| A        | 300 tokens | 50 tokens |
| B        | 800 tokens | 100 tokens |

**Output:** `outputs/brsr_chunks_strategy_{A,B}.jsonl`

In [7]:
def build_full_text_and_spans(pages: pd.DataFrame) -> tuple[str, list[dict[str, int]]]:
    """Concatenate sorted pages into one string; record character spans per page."""
    parts:  list[str]        = []
    spans:  list[dict]       = []
    cursor: int              = 0

    for _, row in pages.sort_values("page_number").iterrows():
        text:   str = row["page_text"] or ""
        pg_num: int = int(row["page_number"])

        start = cursor
        parts.append(text)
        cursor += len(text)
        spans.append({"page_number": pg_num, "start": start, "end": cursor})

        parts.append("\n\n")   # page delimiter
        cursor += 2

    return "".join(parts).rstrip(), spans


def page_range_for_chunk(start: int, end: int, spans: list[dict]) -> tuple[int | None, int | None]:
    """Return (min_page, max_page) whose character span overlaps [start, end)."""
    touched = [
        s["page_number"] for s in spans
        if s["end"] > start and s["start"] < end and (s["end"] - s["start"]) > 0
    ]
    return (min(touched), max(touched)) if touched else (None, None)


def chunk_company(
    company_pages: pd.DataFrame,
    splitter: RecursiveCharacterTextSplitter,
    strategy_name: str,
    chunk_size: int,
    chunk_overlap: int,
) -> list[dict[str, Any]]:
    """Chunk one company's full text and attach metadata."""
    company_name = str(company_pages["company_name"].iloc[0])
    file_name    = str(company_pages["file_name"].iloc[0])
    full_text, spans = build_full_text_and_spans(company_pages)

    docs = splitter.create_documents(
        [full_text],
        metadatas=[{"company_name": company_name, "file_name": file_name}],
    )

    rows: list[dict[str, Any]] = []
    for idx, doc in enumerate(docs):
        text        = doc.page_content or ""
        start_idx   = int(doc.metadata.get("start_index", 0))
        end_idx     = start_idx + len(text)
        pg_start, pg_end = page_range_for_chunk(start_idx, end_idx, spans)

        rows.append({
            "strategy":              strategy_name,
            "target_chunk_tokens":   chunk_size,
            "target_overlap_tokens": chunk_overlap,
            "company_name":          company_name,
            "file_name":             file_name,
            "chunk_index":           idx,
            "page_start":            pg_start,
            "page_end":              pg_end,
            "chunk_char_start":      start_idx,
            "chunk_char_end":        end_idx,
            "chunk_tokens":          token_len(text),
            "chunk_text":            text,
        })
    return rows


def build_all_chunks(pages_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Build chunks for every strategy and return a dict keyed by strategy name."""
    splitters = {
        name: RecursiveCharacterTextSplitter(
            chunk_size=cfg["chunk_size"],
            chunk_overlap=cfg["chunk_overlap"],
            length_function=token_len,
            separators=["\n\n", "\n", " ", ""],
            add_start_index=True,
        )
        for name, cfg in CHUNK_STRATEGIES.items()
    }

    all_rows: list[dict] = []
    for company, grp in tqdm(pages_df.groupby("company_name", sort=True), desc="Chunking"):
        for strat_name, splitter in splitters.items():
            cfg = CHUNK_STRATEGIES[strat_name]
            all_rows.extend(chunk_company(grp, splitter, strat_name, cfg["chunk_size"], cfg["chunk_overlap"]))

    chunks_df = pd.DataFrame(all_rows)

    result: dict[str, pd.DataFrame] = {}
    for strat, sdf in chunks_df.groupby("strategy", sort=True):
        result[strat] = sdf.sort_values(["company_name", "chunk_index"]).reset_index(drop=True)

    print(f"Total chunks : {len(chunks_df)}")
    for strat, sdf in result.items():
        print(f"  Strategy {strat} : {len(sdf)} chunks")
    return result


# ── Run chunking ──────────────────────────────────────────────────────────────
chunk_paths = {strat: OUT_DIR / f"brsr_chunks_strategy_{strat}.jsonl" for strat in CHUNK_STRATEGIES}

if all(p.exists() for p in chunk_paths.values()):
    print("Loading cached chunks …")
    chunks_by_strategy = {strat: pd.read_json(p, lines=True) for strat, p in chunk_paths.items()}
else:
    chunks_by_strategy = build_all_chunks(pages_df)
    for strat, sdf in chunks_by_strategy.items():
        path = chunk_paths[strat]
        sdf.to_json(path, orient="records", lines=True, force_ascii=False)
        print(f"Saved: {path}")

chunks_by_strategy["A"].head(3)

Loading cached chunks …


,strategy,target_chunk_tokens,target_overlap_tokens,company_name,file_name,chunk_index,page_start,page_end,chunk_char_start,chunk_char_end,chunk_tokens,chunk_text
0,A,300,50,ASIANPAINT,ASIANPAINT_24062025100951_BRSRIntimation030620...,0,1,1,0,1055,282,APL/SEC/26/2025-26/14\n3rd June 2025\nBSE Limi...
1,A,300,50,ASIANPAINT,ASIANPAINT_24062025100951_BRSRIntimation030620...,1,1,1,-1,213,61,https://www.asianpaints.com/AnnualReports.html...
2,A,300,50,ASIANPAINT,ASIANPAINT_24062025100951_BRSRIntimation030620...,2,2,2,1113,2666,291,Statutory reports\nBusiness Responsibility and...


---
## 4 · Embedding

Generate dense vector representations for every chunk using **BAAI/bge-small-en-v1.5** (all chunks, both strategies together, used for the shared retrieval demo).  
For the formal benchmark, each setup embeds its own chunks inside `get_setup_assets()`.

**Output:** `outputs/brsr_chunk_embeddings_bge_small.jsonl`

In [9]:
def embed_chunks(
    chunks_df: pd.DataFrame,
    model_name: str = "BAAI/bge-small-en-v1.5",
    batch_size: int = 64,
) -> pd.DataFrame:
    """Embed chunk_text with *model_name* and return a DataFrame with vectors."""
    chunks = chunks_df.copy()
    chunks["chunk_text"] = chunks["chunk_text"].fillna("").astype(str)

    # Derive stable chunk IDs
    if "strategy" in chunks.columns:
        chunks["chunk_id"] = (
            chunks["strategy"].astype(str) + "::" +
            chunks["company_name"].astype(str) + "::" +
            chunks["chunk_index"].astype(str)
        )
    else:
        chunks["chunk_id"] = chunks["company_name"].astype(str) + "::" + chunks["chunk_index"].astype(str)

    def _page_label(row: pd.Series) -> str:
        s, e = row.get("page_start"), row.get("page_end")
        if pd.isna(s) and pd.isna(e):
            return ""
        si = int(s) if not pd.isna(s) else None
        ei = int(e) if not pd.isna(e) else None
        if si is None:
            return str(ei)
        if ei is None:
            return str(si)
        return str(si) if si == ei else f"{si}-{ei}"

    embedder   = SentenceTransformer(model_name)
    embeddings = embedder.encode(
        chunks["chunk_text"].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    out = pd.DataFrame({
        "embedding_vector": [v.tolist() for v in embeddings],
        "chunk_text":       chunks["chunk_text"].values,
        "company":          chunks["company_name"].astype(str).values,
        "page":             chunks.apply(_page_label, axis=1).values,
        "chunk_id":         chunks["chunk_id"].values,
    })
    if "strategy" in chunks.columns:
        out["strategy"] = chunks["strategy"].astype(str).values
    if "file_name" in chunks.columns:
        out["file_name"] = chunks["file_name"].astype(str).values

    return out


# ── Run embedding (all chunks combined, BGE-small) ────────────────────────────
emb_path = OUT_DIR / "brsr_chunk_embeddings_bge_small.jsonl"

if emb_path.exists():
    print(f"Loading cached embeddings from {emb_path}")
    embedding_df = pd.read_json(emb_path, lines=True)
else:
    all_chunks = pd.concat(chunks_by_strategy.values(), ignore_index=True)
    embedding_df = embed_chunks(all_chunks, model_name="BAAI/bge-small-en-v1.5")
    embedding_df.to_json(emb_path, orient="records", lines=True, force_ascii=False)
    print(f"Saved: {emb_path}")

print(f"Embedded chunks : {len(embedding_df)}")
print(f"Vector dim      : {len(embedding_df['embedding_vector'].iloc[0])}")
embedding_df[["chunk_id", "company", "page", "chunk_text"]].head(3)

Loading cached embeddings from /Users/saudquadri/Desktop/Godrej/BRSR_RAG/outputs/brsr_chunk_embeddings_bge_small.jsonl
Embedded chunks : 6894
Vector dim      : 384


,chunk_id,company,page,chunk_text
0,A::ASIANPAINT::0,ASIANPAINT,1,APL/SEC/26/2025-26/14\n3rd June 2025\nBSE Limi...
1,A::ASIANPAINT::1,ASIANPAINT,1,https://www.asianpaints.com/AnnualReports.html...
2,A::ASIANPAINT::2,ASIANPAINT,2,Statutory reports\nBusiness Responsibility and...


---
## 5 · FAISS Index

Build an **IndexFlatIP** (inner-product / cosine) index over the pre-normalised embeddings.

**Output:** `outputs/brsr_faiss_bge_small.index` + `outputs/brsr_faiss_metadata.jsonl`

In [10]:
def parse_vector(x: Any) -> list[float]:
    """Coerce an embedding value (list / ndarray / string) to a Python list."""
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, str):
        return ast.literal_eval(x)
    raise TypeError(f"Unsupported embedding type: {type(x)}")


def build_faiss_index(emb_df: pd.DataFrame) -> tuple[faiss.IndexFlatIP, pd.DataFrame]:
    """Build a FAISS index from *emb_df* and return (index, metadata_df)."""
    vectors = np.array([parse_vector(v) for v in emb_df["embedding_vector"]], dtype=np.float32)
    faiss.normalize_L2(vectors)  # ensure unit-norm for cosine via inner product

    dim   = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)

    meta_cols = [c for c in ["chunk_id", "company", "page", "chunk_text", "strategy", "file_name"] if c in emb_df.columns]
    meta_df   = emb_df[meta_cols].copy().reset_index(drop=True)
    meta_df.insert(0, "faiss_id", np.arange(len(meta_df), dtype=np.int64))

    return index, meta_df


# ── Build / load index ────────────────────────────────────────────────────────
index_path = OUT_DIR / "brsr_faiss_bge_small.index"
meta_path  = OUT_DIR / "brsr_faiss_metadata.jsonl"

if index_path.exists() and meta_path.exists():
    print("Loading cached FAISS index …")
    faiss_index = faiss.read_index(str(index_path))
    meta_df     = pd.read_json(meta_path, lines=True)
else:
    faiss_index, meta_df = build_faiss_index(embedding_df)
    faiss.write_index(faiss_index, str(index_path))
    meta_df.to_json(meta_path, orient="records", lines=True, force_ascii=False)
    print(f"Saved index    : {index_path}")
    print(f"Saved metadata : {meta_path}")

print(f"Vectors indexed : {faiss_index.ntotal}")
print(f"Vector dim      : {faiss_index.d}")
meta_df.head(3)

Loading cached FAISS index …
Vectors indexed : 6894
Vector dim      : 384


,faiss_id,chunk_id,company,page,chunk_text,strategy,file_name
0,0,A::ASIANPAINT::0,ASIANPAINT,1,APL/SEC/26/2025-26/14\n3rd June 2025\nBSE Limi...,A,ASIANPAINT_24062025100951_BRSRIntimation030620...
1,1,A::ASIANPAINT::1,ASIANPAINT,1,https://www.asianpaints.com/AnnualReports.html...,A,ASIANPAINT_24062025100951_BRSRIntimation030620...
2,2,A::ASIANPAINT::2,ASIANPAINT,2,Statutory reports\nBusiness Responsibility and...,A,ASIANPAINT_24062025100951_BRSRIntimation030620...


---
## 6 · Retrieval

All retrieval is **company-scoped**: a per-company sub-index is built on first use and cached.  
Queries are encoded with the same model as the corpus embeddings.



In [11]:
# ── Pre-load shared assets ────────────────────────────────────────────────────
def _load_all_vectors(emb_df: pd.DataFrame) -> np.ndarray:
    vecs = np.array([parse_vector(v) for v in emb_df["embedding_vector"]], dtype=np.float32)
    faiss.normalize_L2(vecs)
    return vecs


_all_vectors: np.ndarray = _load_all_vectors(embedding_df)
_company_index_cache: dict[str, tuple[faiss.IndexFlatIP, np.ndarray]] = {}
_company_to_ids: dict[str, np.ndarray] = {
    company: grp.index.to_numpy(dtype=np.int64)
    for company, grp in meta_df.groupby("company", sort=True)
}
_embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")


def _get_company_index(company: str) -> tuple[faiss.IndexFlatIP, np.ndarray]:
    """Build (and cache) a FAISS sub-index for a single company."""
    if company in _company_index_cache:
        return _company_index_cache[company]
    if company not in _company_to_ids:
        raise ValueError(f"Unknown company '{company}'. Available: {sorted(_company_to_ids)}")
    row_ids         = _company_to_ids[company]
    company_vectors = _all_vectors[row_ids]
    idx             = faiss.IndexFlatIP(company_vectors.shape[1])
    idx.add(company_vectors)
    _company_index_cache[company] = (idx, row_ids)
    return idx, row_ids


def retrieve_top_k(query: str, company: str, top_k: int = TOP_K) -> pd.DataFrame:
    """Return top-*k* chunks for *query* scoped to *company*."""
    idx, row_ids = _get_company_index(company)
    qvec         = _embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, lids = idx.search(qvec, min(top_k, idx.ntotal))

    rows = []
    for score, lid in zip(scores[0], lids[0]):
        if lid < 0:
            continue
        row = meta_df.iloc[int(row_ids[lid])]
        rows.append({
            "score":      float(score),
            "company":    str(row["company"]),
            "page":       str(row["page"]),
            "chunk_id":   str(row["chunk_id"]),
            "chunk_text": str(row["chunk_text"]),
        })
    return pd.DataFrame(rows)


# ── Demo retrieval ────────────────────────────────────────────────────────────
demo_results = retrieve_top_k(
    query="What are the Scope 1 greenhouse gas emissions?",
    company="TCS",
    top_k=5,
)
print(f"Retrieved {len(demo_results)} chunks for TCS")
demo_results

Retrieved 5 chunks for TCS


,score,company,page,chunk_id,chunk_text
0,0.799371,TCS,1,A::TCS::80,Note: Indicate if any independent assessment/e...
1,0.755475,TCS,25,B::TCS::30,151 Business Responsibility & Sustainability R...
2,0.744060,TCS,31,A::TCS::103,157 Business Responsibility & Sustainability R...
3,0.740347,TCS,25,A::TCS::81,Parameter Unit FY 2025 FY 2024\nTotal Scope 1 ...
4,0.738065,TCS,1,B::TCS::31,Total Scope 1 and Scope 2 emission intensity p...


---
## 7 · Ground-Truth & Evaluation Dataset

The ground-truth file (`outputs/ground_truth_raw.json`) contains one structured record per company with ESG fields extracted from the BRSR reports.

We normalise it into a flat table and expand to **20 evaluation questions × N companies**.

**Output:** `outputs/evaluation_normalized_companies.jsonl`, `outputs/evaluation_question_dataset.jsonl`

In [12]:
# ── Helpers for parsing the (sometimes messy) ground-truth JSON ───────────────

def _stringify(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return re.sub(r"\s+", " ", value).strip()
    if isinstance(value, (int, float, bool)):
        return str(value)
    if isinstance(value, list):
        return "; ".join(x for x in (_stringify(v) for v in value) if x)
    if isinstance(value, dict):
        return "; ".join(f"{k}: {_stringify(v)}" for k, v in value.items())
    return str(value)


def _first_nonempty(*values: Any) -> str:
    for v in values:
        s = _stringify(v)
        if s:
            return s
    return ""


def _standardize_units(text: str) -> str:
    t = _stringify(text)
    replacements = {
        r"\bGigajoules?\b":      "GJ",
        r"\bMega\s*Joules?\b":   "MJ",
        r"\bTerajoules?\b":      "TJ",
        r"\bkilolit(?:res|ers)?\b": "kL",
        r"\bmetric\s*ton(?:ne)?s?\b": "metric tonnes",
        r"\bMTCO2e\b":           "tCO2e",
    }
    for pattern, repl in replacements.items():
        t = re.sub(pattern, repl, t, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", t).strip()


def _extract_json_objects(text: str) -> list[dict[str, Any]]:
    """Robustly extract all JSON objects / arrays from a possibly noisy string."""
    decoder = json.JSONDecoder()
    idx, out = 0, []
    while idx < len(text):
        while idx < len(text) and text[idx] not in "[{":
            idx += 1
        if idx >= len(text):
            break
        try:
            obj, end = decoder.raw_decode(text, idx)
        except json.JSONDecodeError:
            idx += 1
            continue
        if isinstance(obj, list):
            out.extend(x for x in obj if isinstance(x, dict))
        elif isinstance(obj, dict):
            out.append(obj)
        idx = end
    return out


def load_ground_truth(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Ground-truth file not found: {path}")
    raw = path.read_text(encoding="utf-8").replace("ground truths-", " ")
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return [x for x in parsed if isinstance(x, dict)]
        if isinstance(parsed, dict):
            return [parsed]
    except Exception:
        pass
    objects = _extract_json_objects(raw)
    if not objects:
        raise ValueError("Ground-truth file contains no parsable JSON.")
    return objects


def normalize_company_record(obj: dict[str, Any]) -> dict[str, str]:
    """Flatten one nested ground-truth record into a single-level dict."""
    ci = obj.get("company_information",   {}) or {}
    bo = obj.get("business_overview",     {}) or {}
    wd = obj.get("workforce_and_diversity", {}) or {}
    em = obj.get("environmental_metrics", {}) or {}
    gp = obj.get("governance_esg_policy", {}) or {}
    wr = obj.get("waste_and_resources",   {}) or {}
    cp = obj.get("compliance",            {}) or {}

    row = {
        "company_name":               _first_nonempty(obj.get("company_name")),
        "cin":                        _first_nonempty(ci.get("corporate_identity_number_cin")),
        "registered_office_address":  _first_nonempty(ci.get("registered_office_address")),
        "financial_year":             _first_nonempty(ci.get("financial_year_covered")),
        "stock_exchanges":            _first_nonempty(ci.get("stock_exchanges_listed_on")),
        "main_business_activities":   _first_nonempty(bo.get("main_business_activities")),
        "top_products":               _first_nonempty(bo.get("top_products_services_by_turnover"),
                                                      bo.get("products_services_contributing_most_to_turnover")),
        "markets_served":             _first_nonempty(bo.get("markets_served")),
        "total_employees":            _first_nonempty(wd.get("total_number_of_employees"),
                                                      wd.get("total_number_of_employees_and_workers")),
        "percentage_women":           _first_nonempty(wd.get("percentage_of_women"),
                                                      wd.get("percentage_of_women_employees"),
                                                      wd.get("percentage_of_employees_who_are_women")),
        "differently_abled_workers":  _first_nonempty(wd.get("employ_differently_abled_workers")),
        "total_energy_consumption":   _first_nonempty(em.get("total_energy_consumption")),
        "scope_1_emissions":          _first_nonempty(em.get("scope_1_greenhouse_gas_emissions"),
                                                      em.get("scope_1_ghg_emissions")),
        "scope_2_emissions":          _first_nonempty(em.get("scope_2_greenhouse_gas_emissions"),
                                                      em.get("scope_2_ghg_emissions")),
        "water_consumption":          _first_nonempty(em.get("water_consumption"),
                                                      em.get("water_consumed"),
                                                      em.get("water_withdrawn"),
                                                      em.get("water_withdrawal_consumed")),
        "sustainability_policy":      _first_nonempty(gp.get("sustainability_or_esg_policy"),
                                                      gp.get("has_sustainability_or_esg_policy")),
        "climate_risk_strategy":      _first_nonempty(gp.get("climate_risk_strategy"),
                                                      gp.get("has_climate_risk_strategy")),
        "board_sustainability_committee": _first_nonempty(gp.get("board_level_committee_for_sustainability")),
        "total_waste_generated":      _first_nonempty(wr.get("total_waste_generated")),
        "waste_recycled_or_reused":   _first_nonempty(wr.get("waste_recycled_or_reused")),
        "external_assurance_brsr":    _first_nonempty(cp.get("external_assurance_for_brsr"),
                                                      cp.get("has_obtained_external_assurance_for_brsr"),
                                                      cp.get("obtained_external_assurance_for_brsr_report")),
    }
    return {k: _standardize_units(v) for k, v in row.items()}


# 20 evaluation questions (question_id, question text, answer_key in normalized record)
QUESTIONS: list[tuple[str, str, str]] = [
    ("Q1",  "What is the Corporate Identity Number (CIN) of the company?",          "cin"),
    ("Q2",  "What is the registered office address of the company?",                "registered_office_address"),
    ("Q3",  "What financial year does this BRSR report cover?",                     "financial_year"),
    ("Q4",  "Which stock exchanges are the company's shares listed on?",            "stock_exchanges"),
    ("Q5",  "What are the main business activities of the company?",               "main_business_activities"),
    ("Q6",  "What products or services contribute most to turnover?",              "top_products"),
    ("Q7",  "What markets does the company serve (national/international)?",       "markets_served"),
    ("Q8",  "What is the total number of employees?",                              "total_employees"),
    ("Q9",  "What percentage of employees are women?",                             "percentage_women"),
    ("Q10", "Does the company employ differently abled workers?",                   "differently_abled_workers"),
    ("Q11", "What is the company's total energy consumption?",                      "total_energy_consumption"),
    ("Q12", "What are the Scope 1 greenhouse gas emissions?",                       "scope_1_emissions"),
    ("Q13", "What are the Scope 2 emissions?",                                      "scope_2_emissions"),
    ("Q14", "How much water was withdrawn or consumed?",                            "water_consumption"),
    ("Q15", "Does the company have a sustainability or ESG policy?",               "sustainability_policy"),
    ("Q16", "Does the company have a climate risk strategy?",                       "climate_risk_strategy"),
    ("Q17", "Is there a board-level committee for sustainability?",                 "board_sustainability_committee"),
    ("Q18", "What is the total waste generated by the company?",                   "total_waste_generated"),
    ("Q19", "How much waste was recycled or reused?",                               "waste_recycled_or_reused"),
    ("Q20", "Has the company obtained external assurance for the BRSR report?",    "external_assurance_brsr"),
]


def build_eval_dataset(normalized_rows: list[dict[str, str]]) -> pd.DataFrame:
    records = []
    for row in normalized_rows:
        for qid, question, answer_key in QUESTIONS:
            records.append({
                "company":             row["company_name"],
                "question_id":         qid,
                "question":            question,
                "ground_truth_answer": row.get(answer_key, ""),
            })
    return pd.DataFrame(records)


def prepare_ground_truth_artifacts() -> tuple[pd.DataFrame, pd.DataFrame]:
    raw        = load_ground_truth(OUT_DIR / "ground_truth_raw.json")
    normalized = [normalize_company_record(x) for x in raw]

    normalized_df = pd.DataFrame(normalized)
    eval_df       = build_eval_dataset(normalized)

    normalized_df.to_json(OUT_DIR / "evaluation_normalized_companies.jsonl", orient="records", lines=True, force_ascii=False)
    eval_df.to_json(OUT_DIR / "evaluation_question_dataset.jsonl",          orient="records", lines=True, force_ascii=False)

    print(f"Companies parsed          : {len(normalized_df)}")
    print(f"Evaluation rows (20×N)    : {len(eval_df)}")
    return normalized_df, eval_df


normalized_df, eval_df = prepare_ground_truth_artifacts()
print("\nNormalized companies (first 3):")
display(normalized_df.head(3))
print("\nEvaluation dataset (first 3 rows):")
display(eval_df.head(3))

Companies parsed          : 5
Evaluation rows (20×N)    : 100

Normalized companies (first 3):


,company_name,cin,registered_office_address,financial_year,stock_exchanges,main_business_activities,top_products,markets_served,total_employees,percentage_women,...,total_energy_consumption,scope_1_emissions,scope_2_emissions,water_consumption,sustainability_policy,climate_risk_strategy,board_sustainability_committee,total_waste_generated,waste_recycled_or_reused,external_assurance_brsr
0,Hero MotoCorp Limited,L35911DL1984PLC017354,"The Grand Plaza, Plot No. 2, Nelson Mandela Ro...","April 1, 2024, to March 31, 2025",National Stock Exchange of India Ltd. (NSE); B...,Manufacturing of two-wheelers (motorcycles and...,product: Motorcycles and scooters; turnover_pe...,"national: 32 states in India (8 plants, 59 off...","4,839 permanent employees (Total workforce of ...",employees: 13.27%; workers: 13.14%,...,"845,800.00 GJ","19,938.00 tCO2e","70,547.00 tCO2e","784,577.00 kL","Yes, policies cover all nine principles of the...",Yes (Goals for 100% Carbon Neutral Operations ...,Yes (Sustainability and Corporate Social Respo...,"19,076.13 metric tonnes","recycled: 24.8 metric tonnes; reused: 5,128 me...",Yes (Reasonable Assurance obtained from Bureau...
1,Larsen & Toubro Limited,L99999MH1946PLC004768,"L&T House, Ballard Estate, Mumbai - 400001, Ma...","April 1, 2024, to March 31, 2025",BSE Limited; National Stock Exchange of India ...,activity: Infrastructure Projects; turnover_pe...,"Construction and maintenance of motorways, str...",national: Pan-India; international: 58 countries,"56,465 (52,505 permanent and 3,960 other-than-...",8.8%,...,"9,930,417 GJ","603,953 tCO2e","282,341 tCO2e","15,431,695 kL","Yes, policies cover each principle and its cor...",Yes (Target for Carbon Neutrality in Scope 1 a...,Yes (CSR & Sustainability Committee of the Board),"451,226 metric tonnes","80,440 metric tonnes (19,475 recycled and 60,9...",Yes (Reasonable assurance for Core KPIs obtain...
2,Dr. Reddy's Laboratories Limited,L85195TG1984PLC004507,"8-2-337, Road No. 3, Banjara Hills, Hyderabad ...","April 1, 2024 to March 31, 2025",BSE Limited; National Stock Exchange of India ...,"Pharmaceuticals (Development, manufacturing & ...","Development, manufacturing & sale of generic f...",The company serves both national (36 States & ...,"34,981 (26,944 permanent and 8,037 other than ...",24% of the total employee workforce.,...,"4,753,823 GJ.","142,772 metric tonnes of CO2 equivalent.","94,690 metric tonnes of CO2 equivalent (market...","1,973,220 kL.","Yes, the entity's policies cover each principl...","Yes, the company is committed to carbon neutra...","Yes, the Sustainability and CSR Committee of t...","112,677.1 metric tonnes.","Recycled: 65,308.02 metric tonnes.; Re-used: 1...","Yes, the company obtained reasonable assurance..."



Evaluation dataset (first 3 rows):


,company,question_id,question,ground_truth_answer
0,Hero MotoCorp Limited,Q1,What is the Corporate Identity Number (CIN) of...,L35911DL1984PLC017354
1,Hero MotoCorp Limited,Q2,What is the registered office address of the c...,"The Grand Plaza, Plot No. 2, Nelson Mandela Ro..."
2,Hero MotoCorp Limited,Q3,What financial year does this BRSR report cover?,"April 1, 2024, to March 31, 2025"


---
## 8 · Retrieval Benchmarking

Each of the 4 setups is evaluated over all (company, question) pairs using:

| Metric | Description |
|--------|-------------|
| **Precision@3** | Fraction of top-3 retrieved chunks that are relevant |
| **Precision@5** | Fraction of top-5 retrieved chunks that are relevant |
| **Recall@5**    | Fraction of all relevant chunks found in top-5 |
| **MRR**         | Mean Reciprocal Rank of the first relevant chunk |

> **Relevance** is defined by cosine similarity between a chunk's embedding and the ground-truth answer embedding exceeding `RELEVANCE_THRESHOLD = 0.45`.

In [13]:
# ── Company name mapper (full name → ticker code) ─────────────────────────────

_COMPANY_ALIAS: dict[str, str] = {
    "heromotocorplimited":              "HEROMOTO",
    "larsentoubrolimited":              "L&T",
    "drreddyslaboratorieslimited":      "DRREDDY",
    "indianoilcorporationlimited":      "IOC",
    "infosyslimited":                   "INFOSYS",
    "oilandnaturalgascorporationlimited": "ONGC",
    "daburindialimited":                "DABURIN",
    "godrejconsumerproductslimited":    "GODREJCP",
    "nhpclimited":                      "NHPC",
    "tataconsultancyserviceslimitedtcs": "TCS",
    "tataconsultancyserviceslimited":   "TCS",
    "hdfcbanklimited":                  "HDFCBANK",
    "relianceindustrieslimited":        "RELIANCE",
    "hcltechnologieslimited":           "HCLTECH",
    "statebankofindia":                 "SBI",
    "nestleindialimited":               "NESTLEIND",
    "britanniaindustrieslimited":       "BRITANNIA",
    "tataconsumerproductslimited":      "TATACP",
    "asianpaintslimited":               "ASIANPAINT",
    "marutisuzukiindialimited":         "MARUTI",
    "suzlonenergylimited":              "SUZLON",
}


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower().replace("&", "and"))


def build_company_mapper(available_codes: list[str]):
    """Return a function: full_name → ticker code (or '' if unknown)."""
    available = set(available_codes)

    def _map(full_name: str) -> str:
        key = _norm(full_name)
        if key in _COMPANY_ALIAS and _COMPANY_ALIAS[key] in available:
            return _COMPANY_ALIAS[key]
        for code in available_codes:
            if _norm(code) in key or key in _norm(code):
                return code
        return ""

    return _map


# ── Per-setup asset management ────────────────────────────────────────────────

def get_setup_assets(setup_name: str, strategy: str, embedding_model: str) -> dict[str, Any]:
    """Build (or load from cache) embeddings + FAISS index for one benchmark setup."""
    setup_dir   = OUT_DIR / "benchmarks" / setup_name
    setup_dir.mkdir(parents=True, exist_ok=True)

    index_path   = setup_dir / "index.faiss"
    meta_path    = setup_dir / "metadata.parquet"
    vectors_path = setup_dir / "vectors.npy"

    if index_path.exists() and meta_path.exists() and vectors_path.exists():
        return {
            "setup_name":      setup_name,
            "strategy":        strategy,
            "embedding_model": embedding_model,
            "index":           faiss.read_index(str(index_path)),
            "meta":            pd.read_parquet(meta_path),
            "vectors":         np.load(vectors_path),
            "setup_dir":       setup_dir,
        }

    # Load the correct chunk strategy
    chunk_path = OUT_DIR / f"brsr_chunks_strategy_{strategy}.jsonl"
    if not chunk_path.exists():
        raise FileNotFoundError(f"Missing chunk file: {chunk_path}")
    chunks = pd.read_json(chunk_path, lines=True).copy()
    chunks["chunk_text"] = chunks["chunk_text"].fillna("").astype(str)
    chunks["company"]    = chunks["company_name"].astype(str)
    chunks["page"]       = chunks.apply(
        lambda r: (
            str(int(r["page_start"])) if pd.notna(r["page_start"]) and pd.notna(r["page_end"]) and int(r["page_start"]) == int(r["page_end"])
            else (f"{int(r['page_start'])}-{int(r['page_end'])}" if pd.notna(r["page_start"]) and pd.notna(r["page_end"]) else "")
        ),
        axis=1,
    )
    chunks["chunk_id"] = chunks["strategy"].astype(str) + "::" + chunks["company_name"].astype(str) + "::" + chunks["chunk_index"].astype(str)

    embedder = SentenceTransformer(embedding_model)
    vectors  = embedder.encode(
        chunks["chunk_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    meta = chunks[["company", "page", "chunk_id", "chunk_text", "strategy"]].copy().reset_index(drop=True)
    meta.insert(0, "faiss_id", np.arange(len(meta), dtype=np.int64))

    faiss.write_index(index, str(index_path))
    meta.to_parquet(meta_path, index=False)
    np.save(vectors_path, vectors)

    return {
        "setup_name": setup_name, "strategy": strategy, "embedding_model": embedding_model,
        "index": index, "meta": meta, "vectors": vectors, "setup_dir": setup_dir,
    }


# ── Retrieval & metrics ───────────────────────────────────────────────────────

def retrieve_for_company(
    query: str,
    company_code: str,
    setup_assets: dict[str, Any],
    embedder: SentenceTransformer,
    top_k: int = TOP_K,
) -> pd.DataFrame:
    meta    = setup_assets["meta"]
    vectors = setup_assets["vectors"]

    company_ids = np.where((meta["company"].astype(str) == str(company_code)).values)[0]
    if len(company_ids) == 0:
        return pd.DataFrame(columns=["score", "company", "page", "chunk_id", "chunk_text", "faiss_id"])

    sub_vecs  = vectors[company_ids].copy()
    sub_index = faiss.IndexFlatIP(sub_vecs.shape[1])
    sub_index.add(sub_vecs)

    qvec   = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, lids = sub_index.search(qvec, min(top_k, sub_index.ntotal))

    rows = []
    for score, lid in zip(scores[0], lids[0]):
        gid  = int(company_ids[lid])
        item = meta.iloc[gid]
        rows.append({
            "score":      float(score),
            "company":    str(item["company"]),
            "page":       str(item["page"]),
            "chunk_id":   str(item["chunk_id"]),
            "chunk_text": str(item["chunk_text"]),
            "faiss_id":   int(item["faiss_id"]),
        })
    return pd.DataFrame(rows)


def relevant_ids_for_answer(
    company_code: str,
    gt_answer: str,
    setup_assets: dict[str, Any],
    embedder: SentenceTransformer,
) -> set[int]:
    """Chunks whose embedding is within RELEVANCE_THRESHOLD cosine-sim of the ground-truth answer."""
    meta    = setup_assets["meta"]
    vectors = setup_assets["vectors"]

    company_ids = np.where((meta["company"].astype(str) == str(company_code)).values)[0]
    if len(company_ids) == 0:
        return set()

    answer_vec   = embedder.encode([gt_answer], normalize_embeddings=True).astype(np.float32)[0]
    sims         = vectors[company_ids] @ answer_vec
    relevant_loc = np.where(sims >= RELEVANCE_THRESHOLD)[0]

    # Always include at least the single most-similar chunk
    if len(relevant_loc) == 0:
        relevant_loc = np.argsort(-sims)[:1]

    return {int(company_ids[i]) for i in relevant_loc}


def retrieval_metrics(hit_ids: list[int], relevant_ids: set[int], k: int) -> tuple[float, float, float]:
    """Return (Precision@k, Recall@k, MRR@k)."""
    if k <= 0 or not relevant_ids:
        return 0.0, 0.0, 0.0
    top         = hit_ids[:k]
    rel_in_top  = sum(1 for h in top if h in relevant_ids)
    precision   = rel_in_top / k
    recall      = rel_in_top / len(relevant_ids)
    first_rank  = next((i for i, h in enumerate(top, 1) if h in relevant_ids), None)
    mrr         = 1.0 / first_rank if first_rank else 0.0
    return precision, recall, mrr


print("Benchmark helpers loaded.")

Benchmark helpers loaded.


In [14]:
# ── Optional LLM generation (Gemini) ─────────────────────────────────────────

def ask_gemini(company: str, question: str, retrieved_df: pd.DataFrame, model) -> str:
    """Generate an answer using Gemini given retrieved chunks as context."""
    blocks = [
        f"[Chunk {i+1} | chunk_id={r['chunk_id']} | page={r['page']} | score={r['score']:.4f}]\n{r['chunk_text']}"
        for i, r in retrieved_df.reset_index(drop=True).iterrows()
    ]
    prompt = f"""\
You are an analyst answering BRSR questions.
Answer only from the retrieved context below. If the answer is not present, write: Not found in retrieved context.

Company: {company}
Question: {question}

Retrieved context:
{chr(10).join(blocks)}

Return:
1) A concise answer
2) Supporting evidence with chunk_id and page number
"""
    response = model.generate_content(prompt)
    return response.text if hasattr(response, "text") else str(response)


# ── Main benchmark runner ─────────────────────────────────────────────────────

def run_single_setup(
    setup_name: str,
    strategy: str,
    embedding_model: str,
    generate_llm_answers: bool = False,
    gemini_model_name: str = DEFAULT_GEMINI_MODEL,
    top_k: int = TOP_K,
) -> dict[str, Any]:
    """
    Run a full retrieval benchmark for one setup configuration.

    Parameters
    ----------
    setup_name          : Identifier used for output paths.
    strategy            : Chunk strategy ('A' or 'B').
    embedding_model     : HuggingFace model name.
    generate_llm_answers: If True, call Gemini for each question (requires GOOGLE_API_KEY).
    top_k               : Number of chunks to retrieve.

    Returns
    -------
    dict with retrieval summary metrics.
    """
    _, eval_df_local = prepare_ground_truth_artifacts()
    setup_assets     = get_setup_assets(setup_name, strategy, embedding_model)
    query_embedder   = SentenceTransformer(embedding_model)

    available_codes  = sorted(setup_assets["meta"]["company"].astype(str).unique())
    mapper           = build_company_mapper(available_codes)

    llm_model = None
    if generate_llm_answers:
        import google.generativeai as genai
        api_key = os.getenv("GOOGLE_API_KEY", "")
        if not api_key:
            raise EnvironmentError("Set GOOGLE_API_KEY to enable LLM generation.")
        genai.configure(api_key=api_key)
        llm_model = genai.GenerativeModel(gemini_model_name)

    run_rows: list[dict] = []
    failures: list[dict] = []

    for _, row in tqdm(eval_df_local.iterrows(), total=len(eval_df_local), desc=setup_name, leave=False):
        company      = str(row["company"])
        company_code = mapper(company)
        question     = str(row["question"])
        ground_truth = str(row["ground_truth_answer"])

        retrieved = retrieve_for_company(question, company_code, setup_assets, query_embedder, top_k)
        hit_ids   = retrieved["faiss_id"].tolist() if not retrieved.empty else []

        rag_answer = ""
        if generate_llm_answers and llm_model:
            rag_answer = ask_gemini(company, question, retrieved, llm_model)

        relevant_ids = relevant_ids_for_answer(company_code, ground_truth, setup_assets, query_embedder)
        p3, _, _     = retrieval_metrics(hit_ids, relevant_ids, 3)
        p5, r5, mrr  = retrieval_metrics(hit_ids, relevant_ids, 5)

        run_rows.append({
            "setup":               setup_name,
            "company":             company,
            "company_code":        company_code,
            "question_id":         row["question_id"],
            "question":            question,
            "ground_truth":        ground_truth,
            "rag_answer":          rag_answer,
            "retrieved_chunk_ids": retrieved["chunk_id"].tolist() if not retrieved.empty else [],
            "retrieval_scores":    retrieved["score"].tolist()    if not retrieved.empty else [],
            "precision_at_3":      p3,
            "precision_at_5":      p5,
            "recall_at_5":         r5,
            "mrr":                 mrr,
        })

        if p5 == 0:
            failures.append({
                "setup":               setup_name,
                "company":             company,
                "question_id":         row["question_id"],
                "failure_type":        "correct_chunk_not_retrieved",
                "question":            question,
                "ground_truth":        ground_truth,
                "retrieved_chunk_ids": run_rows[-1]["retrieved_chunk_ids"],
            })

    run_df     = pd.DataFrame(run_rows)
    failure_df = pd.DataFrame(failures)

    retrieval_summary = {
        "setup":        setup_name,
        "Precision@3":  float(run_df["precision_at_3"].mean()) if len(run_df) else 0.0,
        "Precision@5":  float(run_df["precision_at_5"].mean()) if len(run_df) else 0.0,
        "Recall@5":     float(run_df["recall_at_5"].mean())    if len(run_df) else 0.0,
        "MRR":          float(run_df["mrr"].mean())             if len(run_df) else 0.0,
    }

    # ── Persist per-setup results ─────────────────────────────────────────────
    setup_dir = setup_assets["setup_dir"]
    run_df.to_json(setup_dir / "evaluation_rows.jsonl",    orient="records", lines=True, force_ascii=False)
    failure_df.to_json(setup_dir / "failure_cases.jsonl",  orient="records", lines=True, force_ascii=False)
    with open(setup_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump({"retrieval": retrieval_summary}, f, ensure_ascii=False, indent=2)

    print(f"✓  {setup_name:30s}  P@3={retrieval_summary['Precision@3']:.3f}  "
          f"P@5={retrieval_summary['Precision@5']:.3f}  "
          f"R@5={retrieval_summary['Recall@5']:.4f}  "
          f"MRR={retrieval_summary['MRR']:.3f}  "
          f"failures={len(failure_df)}")

    return {"setup": setup_name, "retrieval": retrieval_summary, "rows": len(run_df), "failures": len(failure_df)}


print("Benchmark runner ready.")

Benchmark runner ready.


---
## 9 · Run All Benchmark Setups

Iterate over the 2 configurations.  
Set `generate_llm_answers=True` to also run Gemini answer generation (requires `GOOGLE_API_KEY`).

In [15]:
all_results: list[dict] = []

for setup_name, cfg in SETUPS.items():
    result = run_single_setup(
        setup_name=setup_name,
        strategy=cfg["strategy"],
        embedding_model=cfg["embedding_model"],
        generate_llm_answers=False,   
    )
    all_results.append(result)

Companies parsed          : 5
Evaluation rows (20×N)    : 100


✓  300_tokens_bge_small            P@3=0.797  P@5=0.796  R@5=0.0234  MRR=0.795  failures=20
Companies parsed          : 5
Evaluation rows (20×N)    : 100


Batches: 100%|██████████| 31/31 [01:02<00:00,  2.03s/it]
                                                                       

✓  800_tokens_bge_small            P@3=0.797  P@5=0.790  R@5=0.0556  MRR=0.795  failures=20


---
## 10 · Results

Aggregate and display retrieval metrics across all setups.  
Final tables are persisted to `outputs/evaluation_retrieval_metrics.jsonl`.

In [18]:
# ── Aggregate retrieval summaries ─────────────────────────────────────────────
summary_rows = []
for setup_name in SETUPS:
    summary_path = OUT_DIR / "benchmarks" / setup_name / "summary.json"
    if summary_path.exists():
        payload = json.loads(summary_path.read_text(encoding="utf-8"))
        summary_rows.append(payload["retrieval"])

retrieval_table = (
    pd.DataFrame(summary_rows)
    .sort_values("setup")
    .reset_index(drop=True)
)

retrieval_table.to_json(OUT_DIR / "evaluation_retrieval_metrics.jsonl", orient="records", lines=True, force_ascii=False)

print("=" * 60)
print("RETRIEVAL EVALUATION RESULTS")
print("=" * 60)
display(
    retrieval_table.style
    .format({"Precision@3": "{:.3f}", "Precision@5": "{:.3f}", "Recall@5": "{:.4f}", "MRR": "{:.3f}"})
    .set_caption("Retrieval metrics across chunk-size × embedding-model configurations")
)

# ── Failure analysis ──────────────────────────────────────────────────────────
failure_parts = []
for setup_name in SETUPS:
    fpath = OUT_DIR / "benchmarks" / setup_name / "failure_cases.jsonl"
    if fpath.exists():
        failure_parts.append(pd.read_json(fpath, lines=True))

failure_df = pd.concat(failure_parts, ignore_index=True) if failure_parts else pd.DataFrame()
failure_df.to_json(OUT_DIR / "evaluation_failure_analysis.jsonl", orient="records", lines=True, force_ascii=False)

print(f"\nTotal failure cases : {len(failure_df)}")
if not failure_df.empty:
    print("\nFailure cases (sample):")
    display(failure_df[["setup", "company", "question_id", "failure_type", "question", "ground_truth"]].head(10))

RETRIEVAL EVALUATION RESULTS


,setup,Precision@3,Precision@5,Recall@5,MRR
0,300_tokens_bge_small,0.797,0.796,0.0234,0.795
1,800_tokens_bge_small,0.797,0.790,0.0556,0.795



Total failure cases : 40

Failure cases (sample):


,setup,company,question_id,failure_type,question,ground_truth
0,300_tokens_bge_small,Larsen & Toubro Limited,Q1,correct_chunk_not_retrieved,What is the Corporate Identity Number (CIN) of...,L99999MH1946PLC004768
1,300_tokens_bge_small,Larsen & Toubro Limited,Q2,correct_chunk_not_retrieved,What is the registered office address of the c...,"L&T House, Ballard Estate, Mumbai - 400001, Ma..."
2,300_tokens_bge_small,Larsen & Toubro Limited,Q3,correct_chunk_not_retrieved,What financial year does this BRSR report cover?,"April 1, 2024, to March 31, 2025"
3,300_tokens_bge_small,Larsen & Toubro Limited,Q4,correct_chunk_not_retrieved,Which stock exchanges are the company's shares...,BSE Limited; National Stock Exchange of India ...
4,300_tokens_bge_small,Larsen & Toubro Limited,Q5,correct_chunk_not_retrieved,What are the main business activities of the c...,activity: Infrastructure Projects; turnover_pe...
5,300_tokens_bge_small,Larsen & Toubro Limited,Q6,correct_chunk_not_retrieved,What products or services contribute most to t...,"Construction and maintenance of motorways, str..."
6,300_tokens_bge_small,Larsen & Toubro Limited,Q7,correct_chunk_not_retrieved,What markets does the company serve (national/...,national: Pan-India; international: 58 countries
7,300_tokens_bge_small,Larsen & Toubro Limited,Q8,correct_chunk_not_retrieved,What is the total number of employees?,"56,465 (52,505 permanent and 3,960 other-than-..."
8,300_tokens_bge_small,Larsen & Toubro Limited,Q9,correct_chunk_not_retrieved,What percentage of employees are women?,8.8%
9,300_tokens_bge_small,Larsen & Toubro Limited,Q10,correct_chunk_not_retrieved,Does the company employ differently abled work...,Yes (68 differently abled employees and 21 dif...


## 11 ·Results Summary & Interpretation

### Retrieval Snapshot
- **300_tokens_bge_small:** Precision@3 = **0.797**, Precision@5 = **0.796**, Recall@5 = **0.0234**, MRR = **0.795**
- **800_tokens_bge_small:** Precision@3 = **0.797**, Precision@5 = **0.790**, Recall@5 = **0.0556**, MRR = **0.795**
- **Total failures logged:** **40**

### Interpretation
- **Top-k quality is strong:** High Precision and MRR (~0.79–0.80) indicate the retriever usually places at least one relevant chunk very high in rank.
- **Coverage is the bottleneck:** Recall@5 is low for both setups, meaning many relevant chunks are still missed within top-5.
- **Chunk-size trade-off:** 800-token chunks improve Recall@5 (0.0556 vs 0.0234) but slightly reduce Precision@5 (0.790 vs 0.796).

### Analysis
- If the goal is **fact coverage/completeness**, prefer **800-token** chunking.
- If the goal is **tight relevance in top hits**, **300-token** chunking is marginally better on precision.
- Most failures are `correct_chunk_not_retrieved`, suggesting next gains should come from retrieval improvements (hybrid retrieval, query expansion/rewrite, reranking) rather than only generation tuning.